In [ ]:
import json

input_file = '../data/data.jsonl'
output_file = '../data/corpus.jsonl'

with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
    for line in infile:
        data = json.loads(line.strip())
        
        doc_id = data.get('_id')
        source = data.get('_source', {})
        title = source.get('name', '')
        text = source.get('searchAttributes', '')

        beir_doc = {
            "_id": doc_id,
            "title": title,
            "text": text
        }

        outfile.write(json.dumps(beir_doc) + '\n')


### STEP 2

In [ ]:
import csv
import json

# Path to your CSV file
input_csv_path = "../data/queries.csv"

# Output path for BEIR format
output_queries_path = "../data/queries.jsonl"

# Read the CSV and convert to BEIR query format
with open(input_csv_path, "r", encoding="utf-8") as f_in, open(output_queries_path, "w", encoding="utf-8") as f_out:
    reader = csv.reader(f_in)
    for i, row in enumerate(reader):
        # Assuming each row has only one column (the query text)
        query_text = row[0].strip()
        if not query_text:
            continue  # skip empty lines

        query_entry = {
            "_id": f"q{i}",
            "text": query_text
        }
        f_out.write(json.dumps(query_entry) + "\n")


### STEP 3

In [ ]:
import csv
import os

input_csv = '../data/keywords.csv'
output_dir = 'qrels'
output_file = os.path.join(output_dir, 'test.tsv')

os.makedirs(output_dir, exist_ok=True)

try:
    with open(input_csv, 'r', encoding='utf-8') as csvfile, open(output_file, 'w', encoding='utf-8') as tsvfile:
        print(f"Opened TSV for writing: {os.path.abspath(output_file)}")
        reader = csv.reader(csvfile)
        header = next(reader)

        for row in reader:
            if len(row) < 3:
                print(f"Skipping short row: {row}")
                continue

            query_id = 'q' + str(int(row[0]) - 1)
            corpus_ids = row[2:]

            for corpus_id in corpus_ids:
                if corpus_id.strip():
                    print(f"Writing: {query_id} -> {corpus_id.strip()}")
                    tsvfile.write(f"{query_id}\t{corpus_id.strip()}\t0\t1\n")

        tsvfile.flush()

except Exception as e:
    print("Error:", e)
